In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Load data
train = pd.read_csv('./train.csv')
test = pd.read_csv('./test.csv')

In [5]:
# Targets and features
targets = ['IC50, mM', 'CC50, mM', 'SI']
X = train.drop(columns=targets + ['index'])
y = train[targets]

In [6]:
# Log transform
y_log = np.log1p(y)
X = X.fillna(0)
test_features = test.drop(columns=['index']).fillna(0)

In [7]:
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y_log, test_size=0.2, random_state=42)

In [8]:
# Train models
models = {}
for i, target in enumerate(targets):
    print(f'Training model for {target}...')

    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='rmse'
    )

    model.fit(X_train, y_train.iloc[:, i], eval_set=[(X_val, y_val.iloc[:, i])], verbose=False)
    models[target] = model

Training model for IC50, mM...
Training model for CC50, mM...
Training model for SI...


In [9]:
# Generate submission
submission = pd.DataFrame({'index': test['index']})

for target in targets:
    model = models[target]
    pred_log = model.predict(test_features)
    pred = np.expm1(pred_log)
    col_name = target.replace(', mM', '') if 'mM' in target else target
    submission[col_name] = pred

submission.to_csv('./submission.csv', index=False)
print('Submission saved!')
print(submission.head())

Submission saved!
   index        IC50        CC50        SI
0      0   50.765785  185.939560  5.230612
1      1   54.250877  322.947052  5.689849
2      2   41.307064  277.597717  6.749296
3      3  147.994522  351.560394  7.772388
4      4   53.645683  188.307007  3.127695
